# Tome 14, Chapter 8: Moments of zeta and random matrices (Python companion)

Exercises first, solutions at the end. We check Ingham's mean square, compute the constants $a_k$ and $g_k$ of the moment conjecture, watch the measured moments creep towards the conjecture, verify the moments of the random Euler product exactly, and see how the Hölder argument of Theorem 8.3 loses its constant. The computer science application: **Monte Carlo with heavy tails**, where the sample mean of $|\zeta|^6$ is a textbook example of an estimator whose error bars lie. Requires `numpy` and `mpmath`.

In [ ]:
import math
import numpy as np
import mpmath
rng = np.random.default_rng(8)
EULER_GAMMA = 0.5772156649015329

def primes_upto(N):
    s = np.ones(N + 1, dtype=bool); s[:2] = False
    for i in range(2, int(N ** 0.5) + 1):
        if s[i]: s[i * i::i] = False
    return np.nonzero(s)[0]

def theta(t):
    return t / 2 * np.log(t / (2 * np.pi)) - t / 2 - np.pi / 8 + 1 / (48 * t) + 7 / (5760 * t ** 3)

def Z_rs(t):
    # Riemann-Siegel formula with its first correction (the notebook of Chapter 7, P7.1)
    t = np.asarray(t, dtype=float)
    a = np.sqrt(t / (2 * np.pi)); N = np.floor(a).astype(int); p = a - N
    th = theta(t); S = np.zeros_like(t)
    for n in range(1, N.max() + 1):
        m = n <= N
        S[m] += np.cos(th[m] - t[m] * math.log(n)) / math.sqrt(n)
    c = np.cos(2 * np.pi * p); c = np.where(np.abs(c) < 1e-9, 1e-9, c)
    return 2 * S + (-1.0) ** (N - 1) * a ** (-0.5) * np.cos(2 * np.pi * (p * p - p - 1 / 16)) / c

def sample_abs_zeta(T, n):
    ts = rng.uniform(T, 2 * T, n)
    A = np.concatenate([np.abs(Z_rs(ts[i:i + 4000])) for i in range(0, n, 4000)])
    return ts, A

P = primes_upto(10 ** 6)

## Exercises

**P8.1 (★) Ingham's formula.** Estimate $\frac1T\int_T^{2T}|\zeta(\frac12+it)|^2\dd t$ for $T=10^3,10^4,10^5$ by averaging over $2\cdot10^5$ random $t$, give a standard error, and compare with the average of $\log\frac t{2\pi}+2\gamma$ (Theorem 8.1). How far off is the cruder $\log T$?

**P8.2 (★) The two constants.** Compute $a_k=\prod_p(1-\frac1p)^{k^2}\sum_r\binom{k+r-1}r^2p^{-r}$ over $p\le10^6$ for $k=\frac12,1,2,3,4$, and check $a_2=6/\pi^2$. Compute $g_k=k^2!\,G(k+1)^2/G(2k+1)$ for $k=1,\dots,5$ with `mpmath.barnesg`, and check that they are integers (Exercises 8.1, 8.2, 8.6).

**P8.3 (★★) The conjecture, measured.** With the samples of P8.1, compute the ratio of the measured $2k$-th moment to the leading term $\frac{a_kg_k}{k^2!}\,\mathbb E\bigl[(\log\frac t{2\pi})^{k^2}\bigr]$ for $k=1,2,3$. The ratios should decrease towards $1$, slowly: the conjecture has lower-order terms of relative size $1/\log T$, many of them.

**P8.4 (★★) Exact moments of the random Euler product.** For $X_p$ independent and uniform on the circle, $\mathbb E|1-X_p/\sqrt p|^{-2k}=\sum_r\binom{k+r-1}r^2p^{-r}$. Compute $\mathbb E\prod_{p\le y}|1-X_p/\sqrt p|^{-2k}$ exactly for $y=10^2,10^3,\dots,10^6$ and $k=1,2,3$, and compare with $a_k(e^\gamma\log y)^{k^2}$ (Proposition 8.4). Then check one case by Monte Carlo.

**P8.5 (★★) The price of Hölder.** For $k=1$ the proof of Theorem 8.3 uses $A(s)=\sum_{n\le x}n^{-s}$ and $\int|\zeta|^2\ge|S_1|^2/S_2$ with $S_1=\int\zeta\overline A$, $S_2=\int|A|^2$. At $T=10^4$, estimate $S_1/T$, $S_2/T$ and the resulting lower bound for $x=T^\theta$, $\theta\in\{0.1,0.2,0.3,0.4\}$, and compare with the true mean square (Exercise 8.5).

**P8.6 (★★) Heavy tails (CS application).** Split $2\cdot10^5$ samples of $|\zeta(\frac12+it)|^6$ at $T=10^4$ into $50$ batches. Compare the spread of the batch means with the standard error that the textbook formula $s/\sqrt n$ gives for one batch, and compute the median-of-means estimate. Which fraction of the sixth moment comes from the top $0.1\%$ of the sample?

## Solutions

### P8.1

In [ ]:
samples = {}
for T in [1e3, 1e4, 1e5]:
    ts, A = sample_abs_zeta(T, 200_000)
    samples[T] = (ts, A)
    m, se = np.mean(A ** 2), np.std(A ** 2) / math.sqrt(len(A))
    ing = np.mean(np.log(ts / (2 * math.pi)) + 2 * EULER_GAMMA)
    print(f"T={T:.0e}: mean |zeta|^2 = {m:.3f} +- {se:.3f}, Ingham {ing:.3f}, log T = {math.log(T):.3f}")

The averages agree with the mean of $\log\frac t{2\pi}+2\gamma$ to within about one standard error at $T=10^4$ and $10^5$, and within three at $T=10^3$, where the $O(T^{0.6})$ error of Theorem 8.1 is not yet small. The cruder $\log T$ overshoots by about $0.3$: the mean of $\log t$ over $[T,2T]$ is $\log T+2\log2-1$, and $2\gamma-\log2\pi\approx-0.68$.

### P8.2

In [ ]:
def a_k(k, R=80):
    tot = 0.0
    for p in P:
        u = 1.0 / float(p); s = 0.0; c = 1.0; term = 1.0
        for r in range(R):
            s += c * c * term
            c *= (k + r) / (r + 1); term *= u
            if term < 1e-18: break
        tot += k * k * math.log1p(-u) + math.log(s)
    return math.exp(tot)

ak = {k: a_k(k) for k in [0.5, 1, 2, 3, 4]}
for k, v in ak.items():
    print(f"a_{k} = {v:.6g}")
print("6/pi^2 =", 6 / math.pi ** 2)
gk = {k: mpmath.barnesg(k + 1) ** 2 / mpmath.barnesg(2 * k + 1) * mpmath.factorial(k * k) for k in range(1, 6)}
print({k: mpmath.nstr(v, 12) for k, v in gk.items()})

$a_2$ agrees with $6/\pi^2=0.60793$ to the shown digits, and $a_{1/2}\approx0.988$ (Exercise 8.6). The $g_k$ come out as $1, 2, 42, 24024, 701149020$: integers, as the formula $g_k=k^2!\prod_{j<k}j!/(j+k)!$ suggests but does not make obvious.

### P8.3

In [ ]:
for T, (ts, A) in samples.items():
    L = np.log(ts / (2 * math.pi))
    out = []
    for k in [1, 2, 3]:
        lead = ak[k] * float(gk[k]) / math.factorial(k * k) * np.mean(L ** (k * k))
        out.append(np.mean(A ** (2 * k)) / lead)
    print(f"T={T:.0e}: ratios k=1,2,3:", np.round(out, 2))

Measured ratios of about $1.2, 5.7, 670$ at $T=10^3$ fall to about $1.1, 2.8, 76$ at $T=10^5$ (Figure 8.2; the $k=3$ value is itself noisy, see P8.6). For $k=1$ the excess is the term $2\gamma-\log2\pi$ relative to $\log t$, a $1/\log T$ effect. For $k=3$ the leading term $\frac{42a_3}{9!}(\log T)^9$ is tiny compared with the eight lower-order terms at these heights: the conjecture is an asymptotic statement, and the full CFKRS polynomial is needed for numerical agreement.

### P8.4

In [ ]:
gam = EULER_GAMMA
for k in [1, 2, 3]:
    row = []
    logE = 0.0; idx = 0
    for y in [1e2, 1e3, 1e4, 1e5, 1e6]:
        while idx < len(P) and P[idx] <= y:
            u = 1.0 / float(P[idx]); s = 0.0; c = 1.0; term = 1.0
            for r in range(80):
                s += c * c * term; c *= (k + r) / (r + 1); term *= u
                if term < 1e-18: break
            logE += math.log(s); idx += 1
        row.append(math.exp(logE) / (ak[k] * (math.exp(gam) * math.log(y)) ** (k * k)))
    print(f"k={k}: ratio E/prediction for y=1e2..1e6:", np.round(row, 4))

# Monte Carlo check: k=1, y=100
ps = P[P <= 100]
X = np.exp(2j * np.pi * rng.random((200_000, len(ps))))
prod = np.prod(np.abs(1 - X / np.sqrt(ps)) ** (-2), axis=1)
exact = np.prod([1 / (1 - 1 / p) for p in ps])
print("Monte Carlo", prod.mean(), "+-", prod.std() / math.sqrt(len(prod)), " exact", exact)

The ratios tend to $1$: the convergence is that of Mertens' theorem, with relative error $O(1/\log y)$, faster than for $\zeta$ itself because there are no zeros to account for. For $k=1$ the exact value is $\prod_{p\le y}(1-\frac1p)^{-1}$, which is Mertens' product itself.

### P8.5

In [ ]:
T = 1e4
ts = rng.uniform(T, 2 * T, 40_000)
# zeta(1/2+it) = e^{-i theta(t)} Z(t)
zeta = np.exp(-1j * theta(ts)) * np.concatenate([Z_rs(ts[i:i + 4000]) for i in range(0, len(ts), 4000)])
true_ms = np.mean(np.abs(zeta) ** 2)
for th in [0.1, 0.2, 0.3, 0.4]:
    x = int(T ** th)
    n = np.arange(1, x + 1)
    Aval = np.array([np.sum(n ** (-0.5 - 1j * t)) for t in ts])
    S1 = np.mean(zeta * np.conj(Aval)); S2 = np.mean(np.abs(Aval) ** 2)
    print(f"theta={th}: x={x}, S1/T={S1.real:.3f}{S1.imag:+.3f}i, S2/T={S2:.3f}, "
          f"bound={abs(S1) ** 2 / S2:.3f}, sum 1/n={np.sum(1 / n):.3f}, true={true_ms:.3f}")

$S_1/T$ and $S_2/T$ both track $\sum_{n\le x}\frac1n\approx\theta\log T+\gamma$, so the bound $|S_1|^2/S_2$ grows like $\theta\log T$: at $\theta=0.4$ it is about $4.5$, against a true mean square of about $9.2$. Cauchy–Schwarz throws away the part of $\zeta$ orthogonal to Dirichlet polynomials of length $x$.

### P8.6

In [ ]:
ts, A = samples[1e4]
w = A ** 6
batches = w.reshape(50, -1)
means = batches.mean(axis=1)
naive_se = batches[0].std() / math.sqrt(batches.shape[1])
print(f"overall mean {w.mean():.1f}; batch means: median {np.median(means):.1f}, "
      f"min {means.min():.1f}, max {means.max():.1f}, sd {means.std():.1f}")
print(f"naive standard error of one batch: {naive_se:.1f}")
srt = np.sort(w)[::-1]
print("share of the top 0.1%:", srt[:len(srt) // 1000].sum() / srt.sum())

At this height the naive error bar of one batch (about $17\,000$) is of the same size as the spread of the batch means (about $15\,500$), so the textbook formula is not yet badly wrong. But the batch means are skewed: the median lies below the mean, the largest batch is about $40\%$ above it, and the naive error bar of a batch is itself random, large exactly when the batch caught a rare huge value. The median of means is more stable but biased low. More than a quarter of the sixth moment sits in the top $0.1\%$ of points (compare Figure 8.1): the moment is carried by rare events, the regime of Exercise 8.3, and the problem gets worse as $T$ or $k$ grows, since the share of the top $0.1\%$ is $62\%$ for $k=3$ at $T=10^6$.